# Saudi Plate OCR — Train a Custom Recognizer (PaddleOCR)

This notebook trains a **specialist** license-plate reader for Saudi plates,
replacing the generic EasyOCR. It reads **both** the Arabic (top) and English
(bottom) halves with a single model.

**Flow:** install → upload our 3 files → generate synthetic data → download a
pretrained Arabic model → fine-tune → export to inference model + ONNX → quick test.

> Runtime: set **Runtime → Change runtime type → GPU (T4)** before running.


## 1. Install dependencies
Fonts (for the data generator), PaddlePaddle-GPU, PaddleOCR repo + package.
If the paddle install fails, check Colab's CUDA with `!nvcc --version` and pick the
matching wheel index from paddlepaddle.org.cn.


In [ ]:
# Arabic + Latin fonts for the generator
!apt-get -qq update
!apt-get -qq -y install fonts-noto-core fonts-kacst fonts-hosny-amiri fonts-dejavu-core > /dev/null

# PaddlePaddle GPU (adjust cuXXX to match Colab CUDA if needed)
!python -m pip install -q paddlepaddle-gpu==2.6.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu123/ || \
  python -m pip install -q paddlepaddle-gpu==2.6.2

# PaddleOCR repo (for tools/ + configs/) pinned to a known-good release
![ -d PaddleOCR ] || git clone -q --depth 1 -b release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git
!python -m pip install -q -r PaddleOCR/requirements.txt
!python -m pip install -q paddleocr paddle2onnx onnxruntime
print('deps installed')


## 2. Get our files from GitHub (private repo)

Clones `saudi-plate-ocr` and copies the 3 source files into the working dir.

**One-time setup:** the repo is private, so add a GitHub token to Colab Secrets:
1. Click the **🔑 key icon** in the left sidebar → **Add new secret**.
2. Name it `GH_TOKEN`, paste a GitHub token with **repo** scope, enable *Notebook access*.
   (Make a token at github.com → Settings → Developer settings → Personal access tokens.)

**To update later:** edit files on your Mac → `git push` → just re-run this cell.


In [ ]:
from google.colab import userdata
import os, subprocess

TOKEN = userdata.get('GH_TOKEN').strip()
REPO  = 'Tanzimalam1454/saudi-plate-ocr'
url   = f'https://{TOKEN}@github.com/{REPO}.git'

subprocess.run(['rm', '-rf', 'saudi-plate-ocr'])
r = subprocess.run(['git', 'clone', '-q', url], capture_output=True, text=True)
print(r.stderr.strip() or 'cloned OK')

for f in ['plate_spec.py', 'generate_plates.py', 'saudi_rec_config.yml']:
    subprocess.run(['cp', f'saudi-plate-ocr/{f}', '.'])

need = ['plate_spec.py', 'generate_plates.py', 'saudi_rec_config.yml']
assert all(os.path.exists(f) for f in need), 'clone/copy failed — check GH_TOKEN secret'
print('files in place:', need)


## 3. Generate synthetic dataset
30k plates → 60k labeled half-crops (Arabic + English). Takes a few minutes.
Bump `--count` for more data; writes `dataset/train_label.txt`, `val_label.txt`,
and `saudi_plate_dict.txt`.


In [ ]:
!python generate_plates.py --count 30000 --out dataset --val-frac 0.08 --montage
!echo '--- sample labels ---'; head -4 dataset/train_label.txt
!echo '--- dict size ---'; wc -l dataset/saudi_plate_dict.txt
from IPython.display import Image as IPImage; IPImage('dataset/_montage.jpg', width=900)


## 4. Download pretrained Arabic recognizer
We fine-tune from this so Arabic + Latin glyphs are already learned. The final
classifier layer is rebuilt for our custom dict automatically.


In [ ]:
import os
os.makedirs('pretrain', exist_ok=True)
URL_V4='https://paddleocr.bj.bcebos.com/PP-OCRv4/multilingual/arabic_PP-OCRv4_rec_train.tar'
URL_V3='https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/arabic_PP-OCRv3_rec_train.tar'
rc = os.system(f'cd pretrain && wget -q {URL_V4} && tar xf arabic_PP-OCRv4_rec_train.tar')
if rc != 0 or not os.path.isdir('pretrain/arabic_PP-OCRv4_rec_train'):
    print('v4 unavailable, falling back to v3'); os.system(f'cd pretrain && wget -q {URL_V3} && tar xf arabic_PP-OCRv3_rec_train.tar')
!ls pretrain


## 5. Train
Fine-tunes the recognizer. Watch `acc` on the eval set climb — synthetic plates
usually reach **>0.95** quickly. Checkpoints land in `output/saudi_rec/`.


In [ ]:
# point config at whichever pretrained model downloaded
import glob, re
pre = glob.glob('pretrain/arabic_PP-OCRv*_rec_train')[0] + '/best_accuracy'
cfg = open('saudi_rec_config.yml').read()
cfg = re.sub(r'pretrained_model:.*', f'pretrained_model: ./{pre}', cfg, count=1)
open('saudi_rec_config.yml','w').write(cfg)
print('pretrained_model ->', pre)

!python PaddleOCR/tools/train.py -c saudi_rec_config.yml


## 6. Evaluate best checkpoint


In [ ]:
!python PaddleOCR/tools/eval.py -c saudi_rec_config.yml \
  -o Global.checkpoints=./output/saudi_rec/best_accuracy


## 7. Export inference model + ONNX
`inference/` is the fast format for serving. The ONNX file is what you'll convert
to TensorRT on the Jetson.


In [ ]:
!python PaddleOCR/tools/export_model.py -c saudi_rec_config.yml \
  -o Global.checkpoints=./output/saudi_rec/best_accuracy \
     Global.save_inference_dir=./inference/saudi_rec

!paddle2onnx --model_dir ./inference/saudi_rec \
  --model_filename inference.pdmodel --params_filename inference.pdiparams \
  --save_file ./inference/saudi_rec.onnx --opset_version 13 --enable_onnx_checker True
!ls -la inference


## 8. Quick sanity test
Run the exported model on a few validation crops and compare to ground truth.


In [ ]:
# Test the exported ONNX directly (the artifact you deploy). Avoids the
# PaddleOCR python API, which changed in 3.x.
import onnxruntime as ort, numpy as np, cv2, random, os

assert os.path.exists('inference/saudi_rec.onnx'), 'run the export cell first'
chars = [c.rstrip('\n').rstrip('\r') for c in open('dataset/saudi_plate_dict.txt')]
chars = [c for c in chars if c != '']
charset = ['<blank>'] + chars + [' ']

sess = ort.InferenceSession('inference/saudi_rec.onnx',
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
inp = sess.get_inputs()[0].name

def prep(img):
    H, W = 48, 320
    h, w = img.shape[:2]
    nw = min(W, max(1, int(H * w / h)))
    r = cv2.resize(img, (nw, H))
    c = np.zeros((H, W, 3), np.float32); c[:, :nw] = r
    return ((c / 255 - 0.5) / 0.5).transpose(2, 0, 1)

def decode(p):
    idx = p.argmax(1); out = []; prev = -1
    for i in idx:
        if i != prev and i != 0: out.append(charset[i])
        prev = i
    return ''.join(out).strip()

val = [l.strip().split('\t') for l in open('dataset/val_label.txt') if l.strip()]
random.shuffle(val); N = 20; ok = 0
for path, gt in val[:N]:
    img = cv2.imread('dataset/' + path)
    out = sess.run(None, {inp: prep(img)[None].astype(np.float32)})[0]
    pred = decode(out[0])
    hit = pred == gt.strip(); ok += hit
    print(('OK ' if hit else 'XX '), f'pred={pred!r:12} gt={gt!r}')
print(f'\n{ok}/{N} exact match')


## 9. Download artifacts
Grab the ONNX, the inference model, and the dict — these go to the Jetson and into
the inference pipeline.


In [ ]:
!cp dataset/saudi_plate_dict.txt inference/
!tar czf saudi_rec_artifacts.tar.gz inference
from google.colab import files; files.download('saudi_rec_artifacts.tar.gz')


## Next steps
- On the Jetson: `trtexec --onnx=saudi_rec.onnx --saveEngine=saudi_rec.engine --fp16`
- Swap EasyOCR for the trained model using `saudi_rec_infer.py` (in `ocr_training/`).
- Once you have **real** drive-thru footage, label a few hundred crops and re-run
  this notebook with them mixed into `dataset/` for a big real-world accuracy bump.
